In [ ]:
using Random
using Statistics
using Printf
using LinearAlgebra
using Plots
using Logging

function find_project_root(start::AbstractString=pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()

include(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "hydrogen_1d", "gfmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

include(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
using .System1D

default(; dpi=170)
nothing


## Model and GFMC Parameters

This notebook runs unguided fixed-population Green Function Monte Carlo for the one-dimensional Coulomb-like potential
`V(x) = -1 / |x|` with open boundaries.

Parameters used below:
- Time step `dt = 2.0e-3`
- Total steps `nsteps = 300`
- Equilibration steps `nequil = 60`
- Target population `targetN = 4000`
- Feedback strength `feedback = 0.15`
- Reconfiguration interval `reconfiguration_interval = 1`
- Branch-weight cap `branch_cap = 4.0`
- ET averaging window `energy_window = 40`
- Initial energy shift `ET0 = -0.5`

Trial / node structure:
- Guiding policy: `NoGuiding()`
- Node policy: `NoNode()`
- Initialization excludes `|x| < x_min` to avoid starting exactly at the singular point


## Julia Construction

The next cell defines the singular Hamiltonian, the nonzero initialization helper, the GFMC parameters, and the notebook toggles.

The CSV filename and debug cadence live there so they can be changed without touching the run cell.


In [ ]:
V(R) = -1 / abs(R[1])
H = Hamiltonian(1, 0.5, V)

function safe_nonzero_x(rng; x_min::Float64=1.0e-2)
    x = randn(rng)
    while abs(x) < x_min
        x = randn(rng)
    end
    return x
end

targetN = 4000
dt = 2.0e-3
nsteps = 300
nequil = 60
ET0 = -0.5
feedback = 0.15
reconfiguration_interval = 1
branch_cap = 4.0
energy_window = 40

params = GFMCParams(dt, nsteps, nequil, targetN, ET0, feedback, reconfiguration_interval, branch_cap, energy_window)
RECONFIGURATION = SystematicReconfiguration()

rng_init = MersenneTwister(1234)
initial_positions = [[safe_nonzero_x(rng_init)] for _ in 1:targetN]

SNAPSHOT_STEPS = nb_default_snapshot_steps(nsteps)
NBINS = 160
DENSITY_SMOOTHING = 9
DENSITY_RANGE = nothing
X_AXIS_LABEL = "x"
PERIOD_MARKERS = nothing

RUN_LABEL = "unguided"
RUN_COLOR = :firebrick
PLOT_TITLE = "Hydrogen 1D GFMC"
DENSITY_TITLE = "Hydrogen 1D GFMC: snapshot densities"
PLOT_MODE = :snapshot_density

SHOW_PROGRESS = false
PROGRESS_EVERY = 0
DEBUG_MODE = false
DEBUG_EVERY = 10
WRITE_RUN_CSV = false
CSV_FILENAME = "hydrogen_1d_unguided_gfmc.csv"
SAVE_FIGURES = false
FIGURE_STEM = "hydrogen_1d_unguided_gfmc"


In [ ]:
sim = GFMCSim(
    H,
    params,
    initial_positions,
    MersenneTwister(42);
    reconfiguration=RECONFIGURATION,
)
run_gfmc!(
    sim;
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])

println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, params.nequil, mean_energy, sem_energy))
println("final fixed walker count = ", sim.population_history[end])
println(@sprintf("final mean weight = %.6f", sim.mean_weight_history[end]))
println(@sprintf("final effective population = %.2f", sim.effective_population_history[end]))

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_gfmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end

SIMS = [sim]
SIM_LABELS = [RUN_LABEL]
SIM_COLORS = [RUN_COLOR]


In [ ]:
history_fig = nb_plot_gfmc_history(SIMS; labels=SIM_LABELS, colors=SIM_COLORS, title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

if DENSITY_RANGE === nothing
    coord_values = Float64[]
    for sim in SIMS
        append!(coord_values, nb_all_coordinates(sim; coord=1))
    end
    xlo, xhi = nb_padded_limits(coord_values; pad_frac=0.12)
else
    xlo, xhi = DENSITY_RANGE
end

density_fig = plot(
    xlabel=X_AXIS_LABEL,
    ylabel="density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(xlo, xhi),
)

if PLOT_MODE == :snapshot_density
    base_sim = SIMS[1]
    available_steps = SNAPSHOT_STEPS[1:min(length(SNAPSHOT_STEPS), length(base_sim.walker_positions_history))]
    for (snapshot, step_idx) in zip(base_sim.walker_positions_history, available_steps)
        centers, density = nb_density_curve_from_snapshot(
            snapshot;
            coord=1,
            nbins=NBINS,
            xmin=xlo,
            xmax=xhi,
            smoothing_window=DENSITY_SMOOTHING,
        )
        plot!(density_fig, centers, density; label="step $(step_idx)", color=SIM_COLORS[1], linewidth=2.2, alpha=0.85)
    end
else
    for (sim, label, color) in zip(SIMS, SIM_LABELS, SIM_COLORS)
        centers, density = nb_density_curve_from_snapshot(
            nb_last_snapshot(sim);
            coord=1,
            nbins=NBINS,
            xmin=xlo,
            xmax=xhi,
            smoothing_window=DENSITY_SMOOTHING,
        )
        plot!(density_fig, centers, density; label=label, color=color, linewidth=2.4)
    end
end

if PERIOD_MARKERS !== nothing
    for (k, xmark) in enumerate(PERIOD_MARKERS)
        vline!(density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
    end
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)
